In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

def ultimo_json(carpeta_base: str) -> str:
    """Devuelve el JSON más reciente por fecha real DD_MM_YYYY.json"""
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)

# ── conectividad ──
f = ultimo_json('../../data/conectividad')
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')


Archivo cargado: ../../data/conectividad/2026/04/13_04_2026.json
Keys disponibles: ['timestamp', 'cobertura_banda_ancha_setid', 'calidad_telecos_cnmc']


In [2]:
#============================================
# Celda 2 — Estructurar metadatos como DataFrame
#============================================
setid = data['cobertura_banda_ancha_setid']
cnmc  = data['calidad_telecos_cnmc']

df_conectividad = pd.DataFrame([
    {
        'fuente':         setid['fuente'],
        'url_datos':      setid['url_datos_municipio'],
        'frecuencia':     setid['frecuencia'],
        'ultimo_informe': setid['ultimo_informe'],
        'tipo_datos':     'cobertura_banda_ancha',
        'granularidad':   'municipio',
        'estado':         'descarga_manual_zip',
        'nota':           setid['nota'],
        'timestamp':      setid['timestamp_captura'],
    },
    {
        'fuente':         cnmc['fuente'],
        'url_datos':      cnmc['url_portal'],
        'frecuencia':     'Trimestral',
        'ultimo_informe': None,
        'tipo_datos':     'calidad_telecos',
        'granularidad':   'operador_ccaa',
        'estado':         cnmc['status'],
        'nota':           cnmc['nota'],
        'timestamp':      cnmc['timestamp_captura'],
    },
])

print(f'Shape: {df_conectividad.shape}')
df_conectividad


Shape: (2, 9)


,fuente,url_datos,frecuencia,ultimo_informe,tipo_datos,granularidad,estado,nota,timestamp
0,SETID - Ministerio para la Transformación Digital,https://avancedigital.mineco.gob.es/banda-anch...,Semestral,2024-S2,cobertura_banda_ancha,municipio,descarga_manual_zip,"ZIP con CSV por municipio: %cobertura fibra, 4...",2026-04-13T11:16:30.996851
1,CNMC - Portal Open Data,https://data.cnmc.es/buscador,Trimestral,NaN,calidad_telecos,operador_ccaa,manual,Sin API pública estable. Descarga manual trime...,2026-04-13T11:16:30.996896


In [3]:
#============================================
# Celda 3 — Datos pendientes de descarga manual
#============================================
print('⚠️  Datos de conectividad requieren descarga manual:')
print()
print('1. SETID — Cobertura banda ancha por municipio')
print('   URL:', setid['url_datos_municipio'])
print('   → Descargar ZIP → extraer CSV → guardar en data/conectividad/setid/')
print()
print('2. CNMC — Calidad telecomunicaciones')
print('   URL:', cnmc['url_portal'])
print('   → Descargar ZIP trimestral → guardar en data/conectividad/cnmc/')
print()
print('Una vez descargados → crear 01_conectividad_datos.ipynb para parsearlos.')


⚠️  Datos de conectividad requieren descarga manual:

1. SETID — Cobertura banda ancha por municipio
   URL: https://avancedigital.mineco.gob.es/banda-ancha/cobertura/Documents/Datos-municipales-BB.zip
   → Descargar ZIP → extraer CSV → guardar en data/conectividad/setid/

2. CNMC — Calidad telecomunicaciones
   URL: https://data.cnmc.es/buscador
   → Descargar ZIP trimestral → guardar en data/conectividad/cnmc/

Una vez descargados → crear 01_conectividad_datos.ipynb para parsearlos.


In [4]:
#============================================
# Celda 4 — Guardar parquet metadatos
#============================================
os.makedirs('../../output/conectividad/01-raw', exist_ok=True)

df_conectividad.to_parquet('../../output/conectividad/01-raw/raw_metadatos_conectividad.parquet', index=False)
print(f'✅ Guardado: output/conectividad/01-raw/raw_metadatos_conectividad.parquet ({len(df_conectividad)} filas)')
print()
print('📌 Pendiente: descargar ZIPs de SETID y CNMC para datos reales.')


✅ Guardado: output/conectividad/01-raw/raw_metadatos_conectividad.parquet (2 filas)

📌 Pendiente: descargar ZIPs de SETID y CNMC para datos reales.
